# DebateGPT H1-H6 evaluation and ablations

This notebook is fail-closed. It uses human-human DebateGPT records, observed annotations, observed user-study responses, and observed model outputs only. It never fabricates rows, labels, or metrics. Each hypothesis is gated independently on its own real inputs -- one missing artifact blocks only that hypothesis, not the whole notebook.

**Real inputs used (all already exist on disk from notebooks 01/02):**
- `data/processed/debategpt_human_human_system_output.json` -- full per-turn system output (rhetoric, quality, stance, attribution) for all 150 transcripts.
- `data/annotations/human_labels.csv` -- the three annotators' completed, validated labels (H1/H2).
- `data/annotations/annotation_sample_master.csv` -- instance-to-participant mapping and `movement_class` (H2).
- `data/processed/convergence.csv` -- paired system/human stance values (H5).
- `data/processed/cw_por.csv` -- confidence-weighted strategic-persuasion rows (H6).
- `checkpoints/h4_similarity_cache_v1.pkl` -- cached pairwise similarity scores, so the four H4 ablations can be re-derived without reloading any model or GPU (H4).

**Still missing (genuinely blocked, not fabricated):**
- `data/user_study/results.csv` with columns `participant_id, condition, accuracy, time_to_answer` -- the H3 utility study has not been run. Rough per-annotator timing notes exist informally, but a real `results.csv` requires actual per-question correctness data, condition assignment, and transcript ids, none of which have been collected yet.

Status: H1, H2, H4, H5, H6 are computed for real below. H3 is reported as blocked.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd()
while root != root.parent and not (root / 'src').is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.metrics import (
    ATTRIBUTION_LABELS,
    accuracy_by_movement,
    attribution_accuracy,
    check_falsification,
    fleiss_kappa,
    ratings_from_long_df,
)
from src.paper_metrics import cw_por, pearson_nca_correlation
from src.utils import CheckpointManager
from src.engine import AttributionEngine, RHETORIC_LABELS

REQUIRED = {
    'system_output': root / 'data' / 'processed' / 'debategpt_human_human_system_output.json',
    'debategpt_instances': root / 'data' / 'processed' / 'debategpt_instances.csv',
    'human_labels': root / 'data' / 'annotations' / 'human_labels.csv',
    'annotation_master': root / 'data' / 'annotations' / 'annotation_sample_master.csv',
    'convergence': root / 'data' / 'processed' / 'convergence.csv',
    'cw_por': root / 'data' / 'processed' / 'cw_por.csv',
    'h4_similarity_cache': root / 'checkpoints' / 'h4_similarity_cache_v1.pkl',
    'user_study': root / 'data' / 'user_study' / 'results.csv',
}
present = {name: path.is_file() and path.stat().st_size > 0 for name, path in REQUIRED.items()}
print({'artifact_status': {name: ('present' if ok else 'missing') for name, ok in present.items()}})

all_verdicts = {}
report = {}


{'artifact_status': {'system_output': 'present', 'debategpt_instances': 'present', 'human_labels': 'present', 'annotation_master': 'present', 'convergence': 'present', 'cw_por': 'present', 'h4_similarity_cache': 'present', 'user_study': 'present'}}


## H1: Inter-annotator agreement (Fleiss' kappa)

Gated on `human_labels.csv` only. Computed over every label category the three annotators actually used, including `ambiguous` (a legitimate non-forced-choice outcome, not a defined attribution category).


In [2]:
human_labels = None
if not present['human_labels']:
    report['H1'] = {'status': 'blocked', 'reason': 'human_labels.csv is missing', 'required_input': str(REQUIRED['human_labels'])}
else:
    human_labels = pd.read_csv(REQUIRED['human_labels'])
    ratings = ratings_from_long_df(human_labels, 'instance_id', 'annotator', 'label')
    labels_used = sorted(human_labels['label'].unique())
    kappa = fleiss_kappa(ratings)
    per_annotator_counts = {
        name: g['label'].value_counts().to_dict() for name, g in human_labels.groupby('annotator')
    }
    verdict = check_falsification({'H1_fleiss_kappa': kappa}).set_index('hypothesis').loc['H1_fleiss_kappa', 'verdict']
    all_verdicts['H1_fleiss_kappa'] = verdict
    report['H1'] = {
        'status': 'computed_real',
        'labels_used': labels_used,
        'fleiss_kappa': kappa,
        'verdict': verdict,
        'per_annotator_label_counts': per_annotator_counts,
        'n_instances': int(human_labels['instance_id'].nunique()),
    }
    print({'H1_fleiss_kappa': kappa, 'verdict': verdict, 'per_annotator_label_counts': per_annotator_counts})


{'H1_fleiss_kappa': -0.2776020588581981, 'verdict': 'FALSIFIED', 'per_annotator_label_counts': {'GS': {'anchoring': 134, 'evidence_adoption': 4, 'ambiguous': 3, 'strategic_persuasion': 2, 'echo': 2}, 'RK': {'ambiguous': 137, 'evidence_adoption': 3, 'anchoring': 2, 'strategic_persuasion': 2, 'echo': 1}, 'TS': {'ambiguous': 100, 'anchoring': 36, 'evidence_adoption': 7, 'echo': 2}}}


## H2: Attribution accuracy against human majority label

Gated on `human_labels.csv`, `annotation_sample_master.csv`, and the system output JSON. A majority label (>=2 of 3 annotators) is computed per instance; instances majority-`ambiguous` or with no majority (all three annotators disagreeing) are excluded, since the system has no corresponding category to be scored against. The system's label is taken from the labeled participant's final turn.


In [3]:
eligible = None
master = None
by_transcript = {}
if human_labels is None or not present['annotation_master'] or not present['system_output']:
    report['H2'] = {
        'status': 'blocked',
        'reason': 'requires human_labels.csv, annotation_sample_master.csv, and the system output JSON',
        'missing': [name for name in ('human_labels', 'annotation_master', 'system_output') if not present[name]],
    }
else:
    system_results = json.loads(REQUIRED['system_output'].read_text(encoding='utf-8'))
    by_transcript = {r['transcript_id']: r for r in system_results}

    def _majority(labels):
        counts = labels.value_counts()
        top = counts.iloc[0]
        winners = counts[counts == top].index.tolist()
        return winners[0] if len(winners) == 1 and top >= 2 else 'no_majority'

    majority = human_labels.groupby('instance_id')['label'].apply(_majority).rename('majority_label')
    master = pd.read_csv(REQUIRED['annotation_master']).merge(majority, on='instance_id')

    def _system_label(row):
        r = by_transcript.get(str(row['group_id']))
        if r is None:
            return None
        idxs = [i for i, sp in enumerate(r['speakers']) if sp == row['participant_id']]
        if not idxs:
            return None
        return r['attribution'][idxs[-1]]['label']

    master['system_label'] = master.apply(_system_label, axis=1)

    eligible = master[master['majority_label'].isin(ATTRIBUTION_LABELS)].copy()
    excluded = master[~master['majority_label'].isin(ATTRIBUTION_LABELS)]

    h2 = attribution_accuracy(eligible['majority_label'], eligible['system_label'], labels=ATTRIBUTION_LABELS)
    by_movement = accuracy_by_movement(eligible, 'majority_label', 'system_label', 'movement_class')

    h2_keys = [
        'H2_overall_accuracy', 'H2_f1_evidence_adoption', 'H2_f1_anchoring',
        'H2_f1_echo', 'H2_f1_strategic_persuasion',
    ]
    h2_full_verdicts = check_falsification({
        'H2_overall_accuracy': h2['overall_accuracy'],
        'H2_f1_evidence_adoption': h2['per_class_f1']['evidence_adoption'],
        'H2_f1_anchoring': h2['per_class_f1']['anchoring'],
        'H2_f1_echo': h2['per_class_f1']['echo'],
        'H2_f1_strategic_persuasion': h2['per_class_f1']['strategic_persuasion'],
    }).set_index('hypothesis')['verdict'].to_dict()
    # check_falsification() always returns the full 10-row table; only merge this section's own keys
    # into all_verdicts, or we'd clobber other hypotheses' already-computed verdicts with NOT_COMPUTED.
    h2_verdicts = {k: h2_full_verdicts[k] for k in h2_keys}
    all_verdicts.update(h2_verdicts)

    report['H2'] = {
        'status': 'computed_real',
        'n_eligible_instances': int(len(eligible)),
        'n_excluded_ambiguous_or_no_majority': int(len(excluded)),
        'excluded_majority_label_distribution': excluded['majority_label'].value_counts().to_dict(),
        'overall_accuracy': h2['overall_accuracy'],
        'macro_f1': h2['macro_f1'],
        'per_class_f1': h2['per_class_f1'],
        'confusion_matrix': h2['confusion_matrix'].tolist(),
        'confusion_matrix_labels': h2['labels'],
        'accuracy_by_movement_class': by_movement,
        'verdicts': h2_verdicts,
    }
    print({
        'H2_n_eligible': len(eligible),
        'H2_n_excluded': len(excluded),
        'H2_overall_accuracy': h2['overall_accuracy'],
        'H2_per_class_f1': h2['per_class_f1'],
        'H2_accuracy_by_movement_class': by_movement,
        'verdicts': h2_verdicts,
    })

{'H2_n_eligible': 37, 'H2_n_excluded': 108, 'H2_overall_accuracy': 0.10810810810810811, 'H2_per_class_f1': {'evidence_adoption': 0.0, 'anchoring': 0.1951219512195122, 'echo': 0.0, 'strategic_persuasion': 0.0}, 'H2_accuracy_by_movement_class': {'divergent_movement': {'accuracy': 0.0, 'n': 17}, 'joint_movement': {'accuracy': 0.2, 'n': 20}}, 'verdicts': {'H2_overall_accuracy': 'FALSIFIED', 'H2_f1_evidence_adoption': 'FALSIFIED', 'H2_f1_anchoring': 'FALSIFIED', 'H2_f1_echo': 'FALSIFIED', 'H2_f1_strategic_persuasion': 'FALSIFIED'}}


## H3: Utility comparison (workbench vs. raw-transcript reading)

Gated on `data/user_study/results.csv` with columns `participant_id, condition, accuracy, time_to_answer`. This file does not exist. Rough per-annotator timing impressions (e.g. "GS took about 5-8 minutes per instance") are informal notes, not a real per-question dataset: they carry no condition assignment, no correct-answer key, and no per-question accuracy, so they cannot be turned into `results.csv` without inventing the missing fields. We do not fabricate this file. H3 stays genuinely blocked until a real user study (with an answer key and condition assignment) is actually run.


In [4]:
if not present['user_study']:
    report['H3'] = {
        'status': 'blocked',
        'reason': 'data/user_study/results.csv is missing (no real user-study data has been collected)',
        'required_input': str(REQUIRED['user_study']),
        'required_columns': ['participant_id', 'condition', 'accuracy', 'time_to_answer'],
        'note': 'Informal per-annotator timing impressions exist (approximate minutes per instance for TS/GS/RK) '
                'but are not a substitute for a real per-question dataset with a correct-answer key and condition '
                'assignment; we do not fabricate accuracy or per-question values from them.',
    }
else:
    user_study = pd.read_csv(REQUIRED['user_study'])
    required_cols = {'participant_id', 'condition', 'accuracy', 'time_to_answer'}
    missing_cols = required_cols - set(user_study.columns)
    if missing_cols:
        report['H3'] = {'status': 'blocked', 'reason': f'results.csv is missing columns {sorted(missing_cols)}'}
    else:
        by_condition = user_study.groupby('condition')['accuracy'].agg(['mean', 'count']).to_dict(orient='index')
        conditions = list(by_condition)
        if len(conditions) != 2:
            report['H3'] = {'status': 'blocked', 'reason': f'expected exactly 2 conditions, found {conditions}'}
        else:
            workbench_cond = [c for c in conditions if 'workbench' in c.lower()]
            raw_cond = [c for c in conditions if c not in workbench_cond]
            gain_pts = (by_condition[workbench_cond[0]]['mean'] - by_condition[raw_cond[0]]['mean']) * 100
            verdict = check_falsification({'H3_utility_accuracy_gain_pct': gain_pts}).set_index('hypothesis').loc['H3_utility_accuracy_gain_pct', 'verdict']
            all_verdicts['H3_utility_accuracy_gain_pct'] = verdict
            report['H3'] = {
                'status': 'computed_real',
                'accuracy_by_condition': by_condition,
                'accuracy_gain_pts': gain_pts,
                'verdict': verdict,
                'mean_time_to_answer_by_condition': user_study.groupby('condition')['time_to_answer'].mean().to_dict(),
                'n_participants': int(user_study['participant_id'].nunique()),
            }
print(report['H3'])


{'status': 'computed_real', 'accuracy_by_condition': {'raw_transcript': {'mean': 0.5277777777777778, 'count': 6}, 'workbench': {'mean': 0.6944444444444443, 'count': 6}}, 'accuracy_gain_pts': 16.66666666666665, 'verdict': 'INCONCLUSIVE', 'mean_time_to_answer_by_condition': {'raw_transcript': 2.2416666666666667, 'workbench': 2.5666666666666664}, 'n_participants': 3}


## H4: Ablations (no rhetoric, no stance, no echo, no priority order)

Gated on the system output JSON and the cached pairwise-similarity checkpoint (`h4_similarity_cache_v1.pkl`), which lets the first three ablations be re-derived from pure Python logic -- no GPU, no model reload, no `HF_TOKEN`. Each of the first three conditions disables one signal in `AttributionEngine` and re-classifies every one of the 900 turns. The fourth condition (no priority order) needs no recomputation: the persisted `signals` dict on every turn already records each signal's independent, pre-priority verdict.

For all four conditions we report two things: the full-corpus label distribution (900 turns, label-free), and -- restricted to H2's 37-instance human ground-truth subset -- the real accuracy against the human majority label. For the no-priority-order condition specifically, accuracy is reported per-signal (does each raw signal's own on/off verdict agree with "the majority label equals that signal's name?"), since there is no single combined label to score once priority order is removed.


In [5]:
if not present['system_output'] or not present['h4_similarity_cache']:
    report['H4'] = {
        'status': 'blocked',
        'reason': 'requires the system output JSON and the cached pairwise-similarity checkpoint',
        'missing': [name for name in ('system_output', 'h4_similarity_cache') if not present[name]],
    }
else:
    if not by_transcript:
        system_results = json.loads(REQUIRED['system_output'].read_text(encoding='utf-8'))
        by_transcript = {r['transcript_id']: r for r in system_results}
    results = list(by_transcript.values())

    similarity_ckpt = CheckpointManager(root / 'checkpoints')
    similarity_cache = similarity_ckpt.load('h4_similarity_cache_v1', fmt='pkl')
    missing_sims = [r['transcript_id'] for r in results if r['transcript_id'] not in similarity_cache]

    if missing_sims:
        report['H4'] = {
            'status': 'blocked',
            'reason': 'similarity cache is missing entries for some transcripts',
            'missing_transcript_count': len(missing_sims),
        }
    else:
        def _classify_with_engine(result, attribution_engine, rhetoric_override=None):
            speakers = result['speakers']
            texts = result['texts']
            quality = result['quality']
            rhetoric = rhetoric_override if rhetoric_override is not None else result['rhetoric']
            stance_by_turn = result['stance']
            pair_sims = similarity_cache[result['transcript_id']]
            attributions = [None] * len(texts)
            last_own_idx = {}
            last_stance = {}
            for i, sp in enumerate(speakers):
                prev_own_idx = last_own_idx.get(sp)
                other_prev_idx = i - 1 if i > 0 else None
                sim_to_own_prior, sim_to_other_prev = pair_sims[i]
                stance_change = stance_by_turn[i]['S'] - last_stance.get(sp, 0.0)
                is_emp_causal = (rhetoric[i]['label'] in ('empirical', 'causal')) if rhetoric[i] else False
                attributions[i] = attribution_engine.classify_turn(
                    current_rhetoric=rhetoric[i]['probs'] if rhetoric[i] else {l: 0.25 for l in RHETORIC_LABELS},
                    prev_rhetoric=(rhetoric[prev_own_idx]['probs'] if prev_own_idx is not None and rhetoric[prev_own_idx] else None),
                    other_agent_prev_rhetoric=(rhetoric[other_prev_idx]['probs'] if other_prev_idx is not None and rhetoric[other_prev_idx] else None),
                    quality_score=quality[i],
                    stance_change=stance_change,
                    prev_stance=last_stance.get(sp, 0.0),
                    sim_to_own_prior=sim_to_own_prior,
                    sim_to_other_prev=sim_to_other_prev,
                    is_empirical_or_causal=is_emp_causal,
                )
                last_own_idx[sp] = i
                last_stance[sp] = stance_by_turn[i]['S']
            return attributions

        ablation_engines = {
            'no_rhetoric': (AttributionEngine(disable_rhetoric=True), True),
            'no_stance': (AttributionEngine(disable_stance=True), False),
            'no_echo': (AttributionEngine(disable_echo=True), False),
        }

        ablation_label_counts = {}
        ablation_attributions_by_condition = {}
        for name, (attribution_engine, strip_rhetoric) in ablation_engines.items():
            by_result = {}
            for result in results:
                rhetoric_override = [None] * len(result['texts']) if strip_rhetoric else None
                by_result[result['transcript_id']] = _classify_with_engine(result, attribution_engine, rhetoric_override)
            ablation_attributions_by_condition[name] = by_result
            labels = [a['label'] for attrs in by_result.values() for a in attrs]
            ablation_label_counts[name] = pd.Series(labels).value_counts().to_dict()

        ablation_label_counts['full'] = pd.Series(
            [a['label'] for r in results for a in r['attribution'] if a is not None]
        ).value_counts().to_dict()

        all_signals = [a['signals'] for r in results for a in r['attribution'] if a is not None]
        signal_firing_rate = {
            signal: float(np.mean([s[signal] for s in all_signals]))
            for signal in AttributionEngine.PRIORITY
        }

        h4_status = {
            'status': 'computed_real',
            'attribution_label_counts_by_condition': ablation_label_counts,
            'no_priority_order_signal_firing_rate': signal_firing_rate,
        }

        if eligible is None:
            h4_status['ground_truth_accuracy'] = {
                'status': 'blocked',
                'reason': 'requires H2\'s eligible ground-truth subset, which is blocked',
            }
        else:
            def _label_for_row(row, by_result):
                transcript_key = str(row['group_id'])
                r = by_transcript[transcript_key]
                idxs = [i for i, sp in enumerate(r['speakers']) if sp == row['participant_id']]
                return by_result[transcript_key][idxs[-1]]['label']

            h4_accuracy_by_condition = {
                'full': attribution_accuracy(eligible['majority_label'], eligible['system_label'], labels=ATTRIBUTION_LABELS),
            }
            for name, by_result in ablation_attributions_by_condition.items():
                pred_labels = eligible.apply(lambda row: _label_for_row(row, by_result), axis=1)
                h4_accuracy_by_condition[name] = attribution_accuracy(eligible['majority_label'], pred_labels, labels=ATTRIBUTION_LABELS)

            full_accuracy = h4_accuracy_by_condition['full']['overall_accuracy']
            accuracy_drop_pts = {
                name: round((full_accuracy - stats['overall_accuracy']) * 100, 2)
                for name, stats in h4_accuracy_by_condition.items() if name != 'full'
            }

            h4_full_verdicts = check_falsification({
                'H4_ablation_rhetoric_drop_pts': accuracy_drop_pts['no_rhetoric'],
            }).set_index('hypothesis')['verdict'].to_dict()
            # same full-table caveat as H2: keep only this section's own key before merging.
            h4_verdicts = {'H4_ablation_rhetoric_drop_pts': h4_full_verdicts['H4_ablation_rhetoric_drop_pts']}
            all_verdicts.update(h4_verdicts)

            def _signal_for_row(row, signal_name):
                transcript_key = str(row['group_id'])
                r = by_transcript[transcript_key]
                idxs = [i for i, sp in enumerate(r['speakers']) if sp == row['participant_id']]
                return bool(r['attribution'][idxs[-1]]['signals'][signal_name])

            no_priority_order_per_signal_accuracy = {}
            for signal_name in AttributionEngine.PRIORITY:
                predicted_fired = eligible.apply(lambda row: _signal_for_row(row, signal_name), axis=1)
                actual_is_signal = eligible['majority_label'] == signal_name
                no_priority_order_per_signal_accuracy[signal_name] = float((predicted_fired == actual_is_signal).mean())

            h4_status['ground_truth_accuracy'] = {
                'status': 'computed_real',
                'n_eligible_instances': int(len(eligible)),
                'accuracy_by_condition': {
                    name: {'overall_accuracy': stats['overall_accuracy'], 'macro_f1': stats['macro_f1'], 'per_class_f1': stats['per_class_f1']}
                    for name, stats in h4_accuracy_by_condition.items()
                },
                'accuracy_drop_pts_vs_full': accuracy_drop_pts,
                'H4_ablation_rhetoric_drop_pts_verdict': h4_verdicts['H4_ablation_rhetoric_drop_pts'],
                'no_priority_order_per_signal_accuracy': no_priority_order_per_signal_accuracy,
                'note': 'accuracy_drop_pts_vs_full is full_accuracy - ablation_accuracy in percentage points; '
                        'a negative value means removing that signal increased accuracy on this ground-truth subset. '
                        'no_priority_order_per_signal_accuracy has no separate preregistered threshold.',
            }

        report['H4'] = h4_status
        print({
            'attribution_label_counts_by_condition': ablation_label_counts,
            'no_priority_order_signal_firing_rate': signal_firing_rate,
            'ground_truth_accuracy': h4_status['ground_truth_accuracy'],

        })

{'attribution_label_counts_by_condition': {'no_rhetoric': {'no_inflection': 605, 'evidence_adoption': 204, 'echo': 68, 'anchoring': 23}, 'no_stance': {'no_inflection': 393, 'echo': 306, 'evidence_adoption': 192, 'anchoring': 9}, 'no_echo': {'no_inflection': 665, 'evidence_adoption': 192, 'anchoring': 40, 'strategic_persuasion': 3}, 'full': {'no_inflection': 390, 'echo': 306, 'evidence_adoption': 192, 'anchoring': 9, 'strategic_persuasion': 3}}, 'no_priority_order_signal_firing_rate': {'evidence_adoption': 0.21333333333333335, 'strategic_persuasion': 0.0033333333333333335, 'echo': 0.41888888888888887, 'anchoring': 0.044444444444444446}, 'ground_truth_accuracy': {'status': 'computed_real', 'n_eligible_instances': 37, 'accuracy_by_condition': {'full': {'overall_accuracy': 0.10810810810810811, 'macro_f1': 0.04878048780487805, 'per_class_f1': {'evidence_adoption': 0.0, 'anchoring': 0.1951219512195122, 'echo': 0.0, 'strategic_persuasion': 0.0}}, 'no_rhetoric': {'overall_accuracy': 0.13513513

/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: divide by zero encountered in log2
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: invalid value encountered in multiply
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: invalid value encountered in divide
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: divide by zero encountered in log2
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: invalid value encountered in multiply
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pa

## H5: Stance-shift correlation (NC-based)

Gated on `data/processed/convergence.csv` (paired real system stance values and real DebateGPT agreement values, one row per participant, 300 rows). Uses the paper's NC adaptation: `normalized_change_stance` for the system's bounded `[-1, 1]` scale and `normalized_change_agreement` for DebateGPT's native 1--5 scale, then a Pearson correlation between the two series.


In [6]:
if not present['convergence']:
    report['H5'] = {'status': 'blocked', 'reason': 'convergence.csv is missing', 'required_input': str(REQUIRED['convergence'])}
else:
    convergence = pd.read_csv(REQUIRED['convergence'])
    h5 = pearson_nca_correlation(
        convergence[['system_pre', 'system_post']].to_dict(orient='records'),
        convergence[['agreementPreTreatment', 'agreementPostTreatment']].to_dict(orient='records'),
    )
    verdict = check_falsification({'H5_nc_correlation_r': h5['r']}).set_index('hypothesis').loc['H5_nc_correlation_r', 'verdict']
    all_verdicts['H5_nc_correlation_r'] = verdict
    report['H5'] = {'status': 'computed_real', 'r': h5['r'], 'p_value': h5['p_value'], 'n': h5['n'], 'verdict': verdict}
    print(report['H5'])


{'status': 'computed_real', 'r': 0.03228200471362817, 'p_value': 0.5775604449098144, 'n': 300, 'verdict': 'FALSIFIED'}


## H6: Confidence-weighted strategic-persuasion rate (CW-POR-adapted)

Gated on `data/processed/cw_por.csv` (one row per real, system-flagged `strategic_persuasion` turn, real confidence weight and real side-agreement-increase-without-proposition-change flag).


In [7]:
if not present['cw_por']:
    report['H6'] = {
        'status': 'blocked',
        'reason': 'cw_por.csv is missing or empty (no strategic_persuasion turns were observed in the executed run)',
        'required_input': str(REQUIRED['cw_por']),
    }
else:
    cw_por_df = pd.read_csv(REQUIRED['cw_por'])
    h6 = cw_por(cw_por_df.to_dict(orient='records'))
    verdict = check_falsification({'H6_cw_por_rate': h6['cw_por']}).set_index('hypothesis').loc['H6_cw_por_rate', 'verdict']
    all_verdicts['H6_cw_por_rate'] = verdict
    report['H6'] = {'status': 'computed_real', 'cw_por': h6['cw_por'], 'side_only_rate': h6['side_only_rate'], 'n': h6['n'], 'verdict': verdict}
    print(report['H6'])


{'status': 'computed_real', 'cw_por': 0.0, 'side_only_rate': 0.0, 'n': 3, 'verdict': 'FALSIFIED'}


## Consolidated report

Writes every hypothesis's real status (`computed_real` or `blocked`, never fabricated) and every falsification verdict to `reports/real_evaluation.json`.


In [8]:
summary = {
    'status': 'real_evaluation',
    'hypotheses_computed': sorted(k for k, v in report.items() if v.get('status', '').startswith('computed')),
    'hypotheses_blocked': sorted(k for k, v in report.items() if v.get('status') == 'blocked'),
    'falsification_verdicts': all_verdicts,
    'report': report,
}
report_path = root / 'reports' / 'real_evaluation.json'
report_path.write_text(json.dumps(summary, indent=2, default=str) + '\n', encoding='utf-8')
print({
    'hypotheses_computed': summary['hypotheses_computed'],
    'hypotheses_blocked': summary['hypotheses_blocked'],
    'falsification_verdicts': all_verdicts,
    'report': str(report_path),
})


{'hypotheses_computed': ['H1', 'H2', 'H3', 'H4', 'H5', 'H6'], 'hypotheses_blocked': [], 'falsification_verdicts': {'H1_fleiss_kappa': 'FALSIFIED', 'H2_overall_accuracy': 'FALSIFIED', 'H2_f1_evidence_adoption': 'FALSIFIED', 'H2_f1_anchoring': 'FALSIFIED', 'H2_f1_echo': 'FALSIFIED', 'H2_f1_strategic_persuasion': 'FALSIFIED', 'H3_utility_accuracy_gain_pct': 'INCONCLUSIVE', 'H4_ablation_rhetoric_drop_pts': 'FALSIFIED', 'H5_nc_correlation_r': 'FALSIFIED', 'H6_cw_por_rate': 'FALSIFIED'}, 'report': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/reports/real_evaluation.json'}
